# 階段 1：NWP 預報風場實驗（Colab Enterprise）

XGBoost 版本已在本機跑完，這本只跑需要 GPU 的模型：
- `nwp_torch`：LSTM 與 Chronos-2，加入 JMA GSM 前一日預報風場
- `nwp_bestmatch`：Chronos-2，加入 Open-Meteo best_match 預報風場（敏感度分析）

1. 連線到 GPU 執行階段，把新的 `pm25_project.zip` 上傳到左側 📁 檔案瀏覽器（覆蓋舊檔）。
2. 按「全部執行」。任何一格出錯都會停下，請截圖或貼最後 30 行給 Claude。
3. 可重跑續跑：已完成的「測站 × 模型」會自動跳過。
4. 最後一格會下載 `pm25_nwp_results.zip`。
5. **跑完務必停止執行階段**。

In [ ]:
%%bash
set -e
# 1. 檢查 GPU 與 Python 版本
nvidia-smi || echo '沒有 GPU，將使用 CPU（較慢但可跑）'
python --version

In [ ]:
# 2. 解壓縮專案（自動尋找上傳的 zip；Colab Enterprise 的檔案瀏覽器根目錄是 /）
import os
candidates = ['/pm25_project.zip', '/content/pm25_project.zip',
              os.path.join(os.getcwd(), 'pm25_project.zip')]
found = [p for p in candidates if os.path.exists(p)]
assert found, '找不到 pm25_project.zip，請先上傳到左側檔案瀏覽器'
os.chdir(os.path.dirname(found[0]))
!rm -rf pm25/src pm25/data/processed
!unzip -q -o pm25_project.zip -d pm25
os.chdir('pm25')
print(os.getcwd())
!ls data/processed | head -3

In [ ]:
%%bash
set -e
# 3. 安裝套件（每次上傳新 zip 後都要重跑；numba 的 numpy 版本警告可忽略）
echo "目前目錄：$(pwd)"
test -f pyproject.toml || { echo "目錄不對：請先重跑第 2 格"; exit 1; }
pip uninstall -q -y pm25-tsfm || true
pip install -q '.[models]'
pip install -q --force-reinstall --no-deps .
cd / && python -c "import pandas, torch; from pm25.models.device import default_device; print('pandas', pandas.__version__, '| torch', torch.__version__, '| device =', default_device())"

## 實驗（每格是獨立行程）

In [ ]:
%%bash
set -e
# 4. LSTM + Chronos-2，JMA GSM 預報風場（L4 預估 20–40 分鐘）
python -m pm25.experiment --family nwp_torch

In [ ]:
%%bash
set -e
# 5. Chronos-2，best_match 預報風場（L4 預估 10–20 分鐘）
python -m pm25.experiment --family nwp_bestmatch

## 打包與下載

In [ ]:
# 6. 確認 36 個預測檔（12 站 × 3 模型）後打包下載
import glob
models = ['lstm_nwp', 'chronos2_wind_nwp', 'chronos2_wind_bestmatch']
files = [f for m in models for f in glob.glob(f'results/predictions/*__{m}.csv.gz')]
print('預測檔數量：', len(files), '/ 36')
assert len(files) == 36, '預測檔不齊全，請往上檢查哪一格出錯'
!rm -f ../pm25_nwp_results.zip
!zip -q ../pm25_nwp_results.zip {' '.join(files)}
!ls -lh ../pm25_nwp_results.zip
from google.colab import files as colab_files
colab_files.download('../pm25_nwp_results.zip')

## 結束
到 Colab Enterprise →「執行階段」把這個執行階段**停止或刪除**，避免繼續計費。